# stoX — TFT Training on Colab GPU

**Before running anything:**
1. Go to **Runtime → Change runtime type → T4 GPU** (free tier) or A100 (Colab Pro)
2. Upload `sl20_feature_panel.parquet` to your Google Drive (any folder is fine)
3. Run cells top to bottom — each section has a ✅ checkpoint

When training finishes, the model checkpoint is saved to your Drive automatically.
You then download `best.ckpt` and drop it into `ml/models/tft_v1/` on your local machine.

> **Note:** Opening this file from GitHub via Colab's file browser only loads the notebook itself — not the rest of the repo. That's fine — Step 2 clones the full repo fresh.

## Step 1 — Verify GPU

In [ ]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\u26a0\ufe0f  No GPU found — go to Runtime \u2192 Change runtime type \u2192 T4 GPU")

## Step 2 — Clone the stoX repo

This always does a **fresh clone** into `/content/stoX_repo` (separate from any
directory Colab may have created when you opened the notebook from GitHub).

In [ ]:
import os
import shutil

# Use a name that won't clash with anything Colab auto-created
REPO_DIR = "/content/stoX_repo"

# Only skip if a REAL git repo already exists here (from a previous run)
if os.path.exists(f"{REPO_DIR}/.git"):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    # Remove the directory if it exists but isn't a real git repo
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    print("Cloning stoX repo...")
    !git clone https://github.com/Dineth-San/stoX.git {REPO_DIR}

# Verify key files are present
checks = [
    f"{REPO_DIR}/ml/pyproject.toml",
    f"{REPO_DIR}/ml/train_model.py",
    f"{REPO_DIR}/ml/configs/pipeline.yaml",
    f"{REPO_DIR}/ml/src/sl20_ml/__init__.py",
]
all_ok = True
for f in checks:
    exists = os.path.exists(f)
    print(f"  {'\u2705' if exists else '\u274c'}  {f.replace(REPO_DIR, '')}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n\u2705 Repo ready at", REPO_DIR)
else:
    print("\n\u274c Some files are missing — re-run this cell")

## Step 3 — Mount Google Drive and copy the parquet file

After mounting, update `PARQUET_IN_DRIVE` to match where you put the file in your Drive.
The default assumes you put it directly in `My Drive` (the root).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── UPDATE THIS if you put the file in a subfolder ────────────────────────────
PARQUET_IN_DRIVE = "/content/drive/MyDrive/sl20_feature_panel.parquet"
# Examples:
#   "/content/drive/MyDrive/stoX/sl20_feature_panel.parquet"
#   "/content/drive/MyDrive/data/sl20_feature_panel.parquet"
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(PARQUET_IN_DRIVE):
    print(f"\u274c File not found: {PARQUET_IN_DRIVE}")
    print("Check the path above — it must match exactly where you uploaded the file in Drive")
else:
    DEST = f"{REPO_DIR}/ml/data/features/sl20_feature_panel.parquet"
    os.makedirs(os.path.dirname(DEST), exist_ok=True)
    shutil.copy(PARQUET_IN_DRIVE, DEST)
    size_mb = os.path.getsize(DEST) / 1e6
    print(f"\u2705 Parquet copied: {size_mb:.1f} MB \u2192 {DEST}")

## Step 4 — Install dependencies

In [ ]:
# Install ML deps (Colab already has torch/numpy/pandas)
!pip install -q pytorch-forecasting>=1.0 lightning>=2.0 mlflow>=2.14 pandera>=0.20 pyyaml>=6.0

# Install the sl20_ml package so its modules are importable
# Primary: editable install from pyproject.toml
result = os.system(f"pip install -q -e {REPO_DIR}/ml")

if result != 0:
    # Fallback: add src/ to path directly (works even without pyproject.toml)
    print("Editable install failed — using sys.path fallback")

import sys
SRC_DIR = f"{REPO_DIR}/ml/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Verify the package is importable
try:
    import sl20_ml
    print("\u2705 sl20_ml importable")
except ImportError as e:
    print(f"\u274c sl20_ml still not importable: {e}")
    print("Re-run this cell, or check that Step 2 succeeded.")

# Quick sanity check on pytorch-forecasting
try:
    import pytorch_forecasting
    import lightning
    print(f"\u2705 pytorch-forecasting {pytorch_forecasting.__version__}")
    print(f"\u2705 lightning {lightning.__version__}")
except ImportError as e:
    print(f"\u274c {e}")

## Step 5 — Set up output directory in Google Drive

The trained model checkpoint will be saved here so it persists after the Colab session ends.

In [ ]:
# Where the checkpoint will be saved on Drive
DRIVE_MODEL_DIR = "/content/drive/MyDrive/stoX_models/tft_v1"
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

# Symlink the model dir inside the repo → Drive
# (so train_model.py saves directly to Drive)
LOCAL_MODEL_DIR = f"{REPO_DIR}/ml/models/tft_v1"
if os.path.islink(LOCAL_MODEL_DIR):
    os.unlink(LOCAL_MODEL_DIR)
elif os.path.isdir(LOCAL_MODEL_DIR):
    shutil.rmtree(LOCAL_MODEL_DIR)

os.makedirs(f"{REPO_DIR}/ml/models", exist_ok=True)
os.symlink(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)

print(f"\u2705 Model output \u2192 {DRIVE_MODEL_DIR}")
print("   (checkpoint saves directly to your Google Drive)")

## Step 6 — Train the model

Expected time on a T4 GPU: **~8–15 minutes** for 30–50 epochs (early stopping kicks in).

Watch the `val_loss` column — it should decrease epoch by epoch. When it stops improving for 15 epochs, training stops automatically.

In [ ]:
import sys

# Ensure src is on path (in case kernel was restarted)
SRC_DIR = f"{REPO_DIR}/ml/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.chdir(f"{REPO_DIR}/ml")

# Run the training script
%run {REPO_DIR}/ml/train_model.py

## Step 7 — Verify the checkpoint was saved

In [ ]:
import json
from pathlib import Path

model_dir = Path(DRIVE_MODEL_DIR)
config_file = model_dir / "model_config.json"

print("Files in model directory:")
for f in sorted(model_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

if config_file.exists():
    cfg = json.loads(config_file.read_text())
    print("\n\u2705 Training complete! Results:")
    print(f"  Val  \u2192 MAE={cfg['val_metrics']['mae']:.4f}  "
          f"DA={cfg['val_metrics']['directional_accuracy']:.1%}  "
          f"QC={cfg['val_metrics']['quantile_coverage']:.1%}")
    print(f"  Test \u2192 MAE={cfg['test_metrics']['mae']:.4f}  "
          f"DA={cfg['test_metrics']['directional_accuracy']:.1%}  "
          f"QC={cfg['test_metrics']['quantile_coverage']:.1%}")
else:
    print("\u26a0\ufe0f  model_config.json not found — training may have failed.")
    print("Check the Step 6 output for error messages.")

## Step 8 — Download the checkpoint to your local machine

**Option A (recommended):** The checkpoint is already in your Google Drive at  
`My Drive / stoX_models / tft_v1 /`. Just download `best.ckpt` and `model_config.json` from [drive.google.com](https://drive.google.com).

**Option B:** Download directly from this notebook (triggers a browser download):

In [ ]:
from google.colab import files

ckpt = next(model_dir.glob("*.ckpt"), None)
if ckpt:
    files.download(str(ckpt))
    print(f"Downloading: {ckpt.name}")
else:
    print("No checkpoint found.")

if config_file.exists():
    files.download(str(config_file))
    print("Downloading: model_config.json")

## Step 9 — What to do with the downloaded files

Once you have `best.ckpt` and `model_config.json` on your local machine:

```
D:\stox\ml\models\tft_v1\best.ckpt          ← drop it here
D:\stox\ml\models\tft_v1\model_config.json   ← drop it here
```

Then test the model locally:
```powershell
cd D:\stox\ml
python predict.py
```

That's it — your locally-served prediction script will use the GPU-trained checkpoint.

---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | In `ml/configs/pipeline.yaml` reduce `batch_size` from 64 → 32 and re-run Step 6 |
| `ModuleNotFoundError: sl20_ml` | Re-run Step 2 (check ✅/❌ for all files), then re-run Step 4 |
| `ModuleNotFoundError: pytorch_forecasting` | Re-run Step 4 |
| `FileNotFoundError: sl20_feature_panel.parquet` | Re-run Step 3 — check the `PARQUET_IN_DRIVE` path matches Drive exactly |
| `File not found at PARQUET_IN_DRIVE` | Open [drive.google.com](https://drive.google.com), right-click the file → Get link, confirm the path |
| Step 2 shows ❌ for some files | The clone failed — re-run Step 2 |
| Session disconnected mid-training | The checkpoint saves every epoch to Drive. Re-run Steps 2–6; training won't restart from scratch (it will load the last saved checkpoint) |

**Runtime tip:** Keep the Colab tab active (open, with the browser visible). Free Colab disconnects after ~90 minutes of inactivity.